# 🔬 Notebook 2: Frequency-Agnostic Acoustic Calibration Lab ($k(f)$ Extraction)
Welcome to the **Acoustic Calibration Protocol Suite** (`v1.1.0`).

This notebook provides a guided calibration wizard to extract the physical acoustic coupling constant $k(f)$ in $\text{V}\cdot\text{m}$, validate inverse-distance decay physics, and export a certified JSON profile (`calibrated_room_profile.json`) for real-time metric distance tracking.

---

### 🏛️ Dual-Regime Calibration Architecture
Depending on your environment, you can switch between two physical calibration protocols with a single toggle:

1. **`MULTIPATH_INDOOR` (`MultipathCalibrationProtocol`):**
   * **Intended environment:** Standard indoor lab benches with walls, ceilings, and table reflections.
   * **Physics:** Solves for the true direct-wave constant $k$ through the **spatial centroid of the interference fringes** in inverse-distance space ($1/r$), avoiding false penalties from standing wave antinodes/nulls.
   * **Diagnostics:** Quantifies Standing Wave Index (SWI) and reflection ripple amplitude.

2. **`FREE_FIELD` (`AcousticCalibrationProtocol`):**
   * **Intended environment:** Open outdoor fields, courtyards, or anechoic chambers.
   * **Physics:** Weighted Least Squares (WLS) regression with dynamic power-law boundary pruning (enforcing $\frac{d\ln V}{d\ln r} \approx -1.0$).

## 1. Initialize Hardware Overlay & Select Calibration Regime
Load the FPGA overlay, define system metadata (speaker volume, mic gain, temperature), and select your calibration regime.

In [ ]:
from pathlib import Path
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from pynq_localizer import (
    MicrophoneArrayOverlay,
    KinematicAnalytics,
    AcousticCalibrationProtocol,
    MultipathCalibrationProtocol,
    AcousticProfile,
    DistanceEstimator
)

# =============================================================================
# 1. Select Calibration Regime
# =============================================================================
CALIBRATION_MODE = "MULTIPATH_INDOOR"  # Options: 'MULTIPATH_INDOOR' or 'FREE_FIELD'

# 2. Initialize FPGA Hardware Overlay
ol = MicrophoneArrayOverlay()

# 3. Traceability Metadata (Keep fixed during calibration)
system_metadata = {
    "speaker_device": "Acoustic_Emitter",
    "speaker_volume_setting": 0.75,          # e.g. 75% volume slider
    "mic_gain_setting": "+35dB / 12 o clock", # Fixed potentiometer position
    "environment_label": "Acoustic_Lab_Bench",
    "temperature_c": 20.0
}

# 4. Instantiate Selected Protocol
if CALIBRATION_MODE.upper() == "MULTIPATH_INDOOR":
    protocol = MultipathCalibrationProtocol(
        r2_threshold=0.80,
        temperature_c=system_metadata["temperature_c"],
        system_metadata=system_metadata
    )
    print("🎯 Active Regime: MULTIPATH_INDOOR (Centroid WLS solver for reflective rooms)")
else:
    protocol = AcousticCalibrationProtocol(
        r2_threshold=0.95,
        system_metadata=system_metadata
    )
    print("🎯 Active Regime: FREE_FIELD (Strict power-law boundary pruner for open/anechoic spaces)")

print(f"✅ Hardware Overlay Active: {ol.current_profile} mode ({ol.fs_per_ch:.0f} SPS per channel)")
print(f"✅ System Metadata Locked: Volume={system_metadata['speaker_volume_setting']*100:.0f}%, Gain={system_metadata['mic_gain_setting']}")

## 2. Carrier Frequency Configuration (Agnostic Multi-Tone / Single-Tone)
The system is completely frequency-agnostic. You can:
* **Auto-Detect:** Listen to the room and lock onto whatever frequency is currently playing.
* **Single Fixed Tone:** Set an exact frequency (e.g. `2660.0 Hz`).
* **Multi-Tone Grid:** Define an array of carrier tones (`[500, 1000, 1500, 2000, 2660, 4000]`) to extract a continuous $k(f)$ spline curve.

In [ ]:
# Mode Selection:
AUTO_DETECT_TONE = True       # True = sample room for loudest tone; False = use manual
MANUAL_CARRIER_FREQ = 2660.0  # Used if AUTO_DETECT_TONE = False

# Multi-Frequency Grid Option (leave as [carrier_freq] for single-tone testing):
# Example for multi-tone: CALIBRATION_FREQUENCIES = [500.0, 1000.0, 1500.0, 2000.0, 2660.0]

if AUTO_DETECT_TONE:
    print("🔍 Sampling acoustic field to auto-detect dominant carrier tone...")
    test_frame = ol.capture_quadruple(source="A0", f_min=100.0, f_max=15000.0, timeout=1.0)
    detected_pitch = test_frame["quadruple"]["frequency_hz"]
    detected_amp = test_frame["quadruple"]["amplitude_v"]
    
    if np.isfinite(detected_pitch) and detected_amp >= 0.003:
        carrier_freq = float(np.round(detected_pitch, 1))
        print(f"🎯 LOCKED onto dominant tone: {carrier_freq:.1f} Hz ({detected_amp*1000:.1f} mV RMS)")
    else:
        carrier_freq = float(MANUAL_CARRIER_FREQ)
        print(f"⚠️ No active tone detected above noise floor. Using manual fallback: {carrier_freq:.1f} Hz")
else:
    carrier_freq = float(MANUAL_CARRIER_FREQ)
    print(f"🎯 Manual carrier frequency locked: {carrier_freq:.1f} Hz")

# Active calibration frequency list:
CALIBRATION_FREQUENCIES = [carrier_freq]
print(f"   Active Calibration Frequencies ({len(CALIBRATION_FREQUENCIES)} tone(s)): {CALIBRATION_FREQUENCIES} Hz")

## 3. Interactive Guided Calibration Wizard (Single Automated Cell)
Place your measuring tape along the line of sight. Run the cell below:
1. It prompts you to position the speaker at each distance in `calibration_distances_m`.
2. Move the speaker to the distance mark, then press **`[Enter]`**.
3. It automatically bursts $N=30$ frames per frequency, aggregates statistics, and advances to the next distance.
4. When all stations are measured, it automatically executes the regression solver and renders interactive Plotly diagnostic graphs.

In [ ]:
import time

# Define physical distance stations in meters
calibration_distances_m = [
    0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00
]

protocol.clear()
n_burst_samples = 30

print("=" * 78)
print(f"🚀 STARTING GUIDED CALIBRATION WIZARD ({CALIBRATION_MODE})")
print(f"   • Distance Stations ({len(calibration_distances_m)} pts) : {[f'{d*100:.0f}cm' for d in calibration_distances_m]}")
print(f"   • Frequencies       ({len(CALIBRATION_FREQUENCIES)} tones) : {CALIBRATION_FREQUENCIES} Hz")
print(f"   • Bursts Per Station : N = {n_burst_samples} samples/tone")
print("=" * 78)

# Guided interactive loop
for idx, dist_m in enumerate(calibration_distances_m):
    dist_cm = dist_m * 100.0
    input(f"\n📍 [{idx+1}/{len(calibration_distances_m)}] Place speaker at {dist_cm:5.1f} cm ({dist_m:.2f} m) and press [Enter]...")
    
    for f_target in CALIBRATION_FREQUENCIES:
        f_min = max(20.0, f_target - 50.0)
        f_max = f_target + 50.0
        
        samples = []
        for _ in range(n_burst_samples):
            frame = ol.capture_quadruple(source="A0", f_min=f_min, f_max=f_max, timeout=0.35)
            amp_v = frame["quadruple"]["amplitude_v"]
            if amp_v > 0.0005:
                samples.append(amp_v)
            time.sleep(0.005)
            
        if len(samples) < 5:
            print(f"    ⚠️ f={f_target:.1f} Hz: Weak signal at {dist_cm:.1f} cm! Check emitter.")
            continue
            
        protocol.add_measurement(distance_m=dist_m, frequency_hz=f_target, amplitude_v=samples)
        mean_v = float(np.mean(samples))
        std_v = float(np.std(samples, ddof=1)) if len(samples) > 1 else 0.0
        sem_v = std_v / np.sqrt(len(samples))
        print(f"    ✅ f={f_target:6.1f} Hz: V_RMS = {mean_v*1000.0:6.2f} mV ± {sem_v*1000.0:4.2f} mV (σ={std_v*1000.0:4.2f} mV, N={len(samples)})")

print("\n" + "=" * 78)
print(f"📊 ALL STATIONS MEASURED — SOLVING {CALIBRATION_MODE} REGRESSIONS...")
print("=" * 78)

# Execute Protocol Fit
fit_results = protocol.fit()
valid_freqs = sorted(fit_results.keys())

# Summary Table
for f in valid_freqs:
    res = fit_results[f]
    swi_str = f" | SWI = {res['standing_wave_index']:.2f}" if "standing_wave_index" in res else ""
    print(f"\n🏆 Results for f = {f:.1f} Hz:")
    print(f"   • Coupling Constant k   : {res['k']:.4f} ± {res['delta_k']:.4f} V·m")
    print(f"   • Room Reverberation c  : {res['c_room']*1000.0:.2f} mV")
    print(f"   • Goodness-of-Fit R²    : {res['r_squared']:.4f} (Gate >= {protocol.r2_threshold}){swi_str}")
    print(f"   • Passed Quality Gate   : {'✅ YES' if res['passed_gate'] else '❌ NO'}")
    print(f"   • Active Distance Range : [{res['r_valid_min_m']*100:.0f} cm -> {res['r_valid_max_m']*100:.0f} cm] ({res['n_pruned_points']}/{res['n_total_points']} stations)")

# -----------------------------------------------------------------------------
# Adaptive Diagnostic Plotting
# -----------------------------------------------------------------------------
if len(valid_freqs) == 1:
    # Single Tone Diagnostics: Linear 1/r fit (Panel 1) + Physical Decay Curve (Panel 2)
    f_single = valid_freqs[0]
    res = fit_results[f_single]
    stations = res["raw_stations"]
    
    r_all_m = np.array([st["r_m"] for st in stations])
    v_all_mv = np.array([st["mean_v"] for st in stations]) * 1000.0
    err_all_mv = np.array([st["std_v"] for st in stations]) * 1000.0
    is_active = np.array([st["is_pruned_in"] for st in stations])
    
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"<b>1. Linear Domain: V_RMS vs 1/r (f0 = {f_single:.1f} Hz, R² = {res['r_squared']:.4f})</b>",
            f"<b>2. Physical Domain: V_RMS vs Distance ({r_all_m.min()*100:.0f} – {r_all_m.max()*100:.0f} cm)</b>"
        ),
        horizontal_spacing=0.10
    )
    
    # Panel 1: Linear 1/r fit
    fig.add_scatter(x=1.0 / r_all_m[is_active], y=v_all_mv[is_active], error_y=dict(type="data", array=err_all_mv[is_active], visible=True), mode="markers", marker=dict(size=9, color="#00FFCC"), name="Active Stations", row=1, col=1)
    if np.any(~is_active):
        fig.add_scatter(x=1.0 / r_all_m[~is_active], y=v_all_mv[~is_active], error_y=dict(type="data", array=err_all_mv[~is_active], visible=True), mode="markers", marker=dict(size=7, color="gray", symbol="x"), name="Pruned Stations", row=1, col=1)
    
    inv_r_line = np.linspace(1.0 / r_all_m.max() * 0.85, 1.0 / r_all_m.min() * 1.05, 100)
    v_line = (res["k"] * inv_r_line + res["c_room"]) * 1000.0
    fig.add_scatter(x=inv_r_line, y=v_line, mode="lines", line=dict(color="#00FFCC", dash="dash", width=2), name="Centroid Model Line", row=1, col=1)
    
    # Panel 2: Physical distance decay [cm]
    r_dense_m = np.linspace(r_all_m.min() * 0.9, r_all_m.max() * 1.05, 150)
    v_phys_curve = (res["k"] / r_dense_m + res["c_room"]) * 1000.0
    fig.add_scatter(x=r_dense_m * 100.0, y=v_phys_curve, mode="lines", line=dict(color="#FFA500", width=2), name="Direct k/r Decay", row=1, col=2)
    fig.add_scatter(x=r_all_m[is_active] * 100.0, y=v_all_mv[is_active], error_y=dict(type="data", array=err_all_mv[is_active], visible=True), mode="markers", marker=dict(size=9, color="#00FFCC"), name="Measured Points", row=1, col=2, showlegend=False)
    if np.any(~is_active):
        fig.add_scatter(x=r_all_m[~is_active] * 100.0, y=v_all_mv[~is_active], error_y=dict(type="data", array=err_all_mv[~is_active], visible=True), mode="markers", marker=dict(size=7, color="gray", symbol="x"), name="Pruned Points", row=1, col=2, showlegend=False)
    
    fig.update_layout(template="plotly_dark", height=460, title=f"<b>Acoustic Calibration Laboratory ({CALIBRATION_MODE})</b>")
    fig.update_xaxes(title="1 / Distance [m⁻¹]", row=1, col=1)
    fig.update_yaxes(title="In-Band V_RMS [mV]", row=1, col=1)
    fig.update_xaxes(title="Physical Distance [cm]", row=1, col=2)
    fig.update_yaxes(title="In-Band V_RMS [mV]", row=1, col=2)
    fig.show()
else:
    # Multi-Tone Diagnostics: Linear fits (Panel 1) + Continuous k(f) Spline (Panel 2) + R² Quality Gate (Panel 3)
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=(
            "<b>1. Multi-Tone Linear Fits</b>",
            "<b>2. Calibrated k(f) Curve [V·m]</b>",
            "<b>3. Goodness-of-Fit R² Gate</b>"
        ),
        horizontal_spacing=0.08
    )
    k_list = [fit_results[f]["k"] for f in valid_freqs]
    k_err_list = [fit_results[f]["delta_k"] for f in valid_freqs]
    r2_list = [fit_results[f]["r_squared"] for f in valid_freqs]
    
    colors = ["#00FFCC", "#FFA500", "#00E5FF", "#FF007F", "#76FF03", "#E040FB"]
    for idx, f in enumerate(valid_freqs):
        res = fit_results[f]
        st = res["raw_stations"]
        r_m = np.array([s["r_m"] for s in st if s["is_pruned_in"]])
        v_mv = np.array([s["mean_v"] for s in st if s["is_pruned_in"]]) * 1000.0
        c = colors[idx % len(colors)]
        if len(r_m) > 0:
            fig.add_scatter(x=1.0 / r_m, y=v_mv, mode="markers", marker=dict(size=7, color=c), name=f"{f:.0f} Hz", row=1, col=1)
            inv_line = np.linspace(min(1.0/r_m)*0.9, max(1.0/r_m)*1.1, 40)
            fig.add_scatter(x=inv_line, y=(res["k"]*inv_line + res["c_room"])*1000.0, mode="lines", line=dict(color=c, dash="dash"), showlegend=False, row=1, col=1)
            
    # Panel 2: Continuous k(f)
    fig.add_scatter(x=valid_freqs, y=k_list, error_y=dict(type="data", array=k_err_list, visible=True), mode="lines+markers", line=dict(color="#00FFCC", width=2), marker=dict(size=8), name="k(f)", row=1, col=2)
    # Panel 3: R^2 Gate
    fig.add_bar(x=[f"{f:.0f}" for f in valid_freqs], y=r2_list, marker=dict(color="#00E5FF"), name="R² Score", row=1, col=3)
    fig.add_hline(y=protocol.r2_threshold, line=dict(color="orange", dash="dash"), annotation_text=f"R² Gate ({protocol.r2_threshold})", row=1, col=3)
    
    fig.update_layout(template="plotly_dark", height=460, title="<b>Multi-Tone Acoustic Calibration Suite</b>")
    fig.update_xaxes(title="1 / Distance [m⁻¹]", row=1, col=1)
    fig.update_yaxes(title="In-Band V_RMS [mV]", row=1, col=1)
    fig.update_xaxes(title="Frequency [Hz]", row=1, col=2)
    fig.update_yaxes(title="k [V·m]", row=1, col=2)
    fig.update_xaxes(title="Frequency [Hz]", row=1, col=3)
    fig.update_yaxes(title="R² Score", range=[0.5, 1.02], row=1, col=3)
    fig.show()

## 4. Export Certified Profile to JSON
Serialize the calibration results to `calibrated_room_profile.json`. This stores the continuous evaluated $k(f)$, measurement uncertainties, operating bounds, and reflection metadata.

> **Note:** Notebook 1 (`01_realtime_kinematics_telemetry.ipynb`) and `KinematicsDashboard` will automatically detect and load this profile file!

In [ ]:
export_path = "calibrated_room_profile.json"
saved_file = protocol.save_profile_json(
    filepath=export_path,
    name=f"Room_Calibrated_Profile_{CALIBRATION_MODE}",
    description=f"Calibrated profile via {CALIBRATION_MODE} with N={n_burst_samples} statistical bursts"
)

print(f"🎉 SUCCESS: Saved calibrated profile to: {saved_file.resolve()}")
print("   Notebook 1 is now ready to track physical distances with certified room calibration!")

## 5. Live Single-Channel Distance Inversion Verification & Hardware Release
Load the profile into runtime `DistanceEstimator` and test metric distance tracking in centimeters.

In [ ]:
# Load generated profile into DistanceEstimator
loaded_profile = AcousticProfile.from_json(export_path)
estimator = DistanceEstimator(profile=loaded_profile, noise_gate_v=0.003)

input("\n📍 Move your speaker to an arbitrary distance (e.g. ~45 cm) and press [Enter] to test...")

test_f = CALIBRATION_FREQUENCIES[0]
frame = ol.capture_quadruple(source="A0", f_min=test_f - 50.0, f_max=test_f + 50.0, timeout=0.5)
result = estimator.process_frame(frame, source="A0")

print("\n--- Live Metric Distance Inversion Result ---")
print(f"Detected Pitch f0  : {result['frequency_hz']:.1f} Hz")
print(f"In-Band Amplitude  : {result['amplitude_v']*1000.0:.2f} mV RMS")
print(f"Evaluated k(f0)    : {result['k_evaluated']:.4f} V·m")
print(f"Inverted Distance  : {result['distance_m']*100.0:.1f} cm (±{result['distance_err_m']*100.0:.1f} cm)")
print(f"Operational Status : {result['distance_status']}")

# Cleanly release FPGA DMA buffers
ol.close()
print("\n🔒 Hardware resources cleanly released.")